## Imports Libraries

In [ ]:
# =======================================================
# Imports 
# =======================================================

import os
import json
from urllib.parse import quote_plus

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sqlalchemy import create_engine, text

## Configuration

In [ ]:
# ============================================================
# Project Paths
# ============================================================

PROJECT_HOME = "/home/bibek-karki/mlops"

DATA_PATH = PROJECT_HOME + "/data/body_performance.csv"

REPORT_DIR = PROJECT_HOME + "/reports/eda"
FIGURE_DIR = REPORT_DIR + "/figures"

os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

# ============================================================
# MariaDB ColumnStore Configuration
# ============================================================

DB_USER = "mariadbuser"
DB_PASSWORD = quote_plus("Sunway@123")
DB_HOST = "127.0.0.1"
DB_PORT = 3307
DB_NAME = "body_performance_db"

OBT_TABLE_NAME = "body_performance_obt"
PREDICTION_LOG_TABLE = "prediction_log"

DEFAULT_CONN_STRING = (
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# ============================================================
# Dataset Columns
# ============================================================

RAW_COLUMNS = [
    "age",
    "gender",
    "height_cm",
    "weight_kg",
    "body fat_%",
    "diastolic",
    "systolic",
    "gripForce",
    "sit and bend forward_cm",
    "sit-ups counts",
    "broad jump_cm",
    "class",
]

RENAME_MAP = {
    "body fat_%": "body_fat_percent",
    "gripForce": "grip_force",
    "sit and bend forward_cm": "sit_and_bend_forward_cm",
    "sit-ups counts": "sit_ups_counts",
    "broad jump_cm": "broad_jump_cm",
    "class": "fitness_class",
}

FEATURE_COLUMNS = [
    "age",
    "gender",
    "height_cm",
    "weight_kg",
    "body_fat_percent",
    "diastolic",
    "systolic",
    "grip_force",
    "sit_and_bend_forward_cm",
    "sit_ups_counts",
    "broad_jump_cm",
]

NUMERIC_FEATURE_COLUMNS = [
    "age",
    "height_cm",
    "weight_kg",
    "body_fat_percent",
    "diastolic",
    "systolic",
    "grip_force",
    "sit_and_bend_forward_cm",
    "sit_ups_counts",
    "broad_jump_cm",
]

CATEGORICAL_COLUMNS = [
    "gender",
    "fitness_class",
]

TARGET_COLUMN = "fitness_class"
TARGET_ORDER = ["A", "B", "C", "D"]

VALID_RANGES = {
    "age": (10, 100),
    "height_cm": (100, 250),
    "weight_kg": (20, 200),
    "body_fat_percent": (1, 60),
    "diastolic": (40, 130),
    "systolic": (60, 220),
    "grip_force": (0, 100),
    "sit_and_bend_forward_cm": (-50, 250),
    "sit_ups_counts": (0, 100),
    "broad_jump_cm": (0, 400),
}

INTEGER_COLUMNS = [
    "age",
    "systolic",
    "diastolic",
    "sit_ups_counts",
    "broad_jump_cm",
]

print("Configuration loaded successfully.")

## Load Stored OBT Data from MariaDB

In [ ]:
# ============================================================
# Load Stored OBT Data from MariaDB
# ============================================================

engine = create_engine(DEFAULT_CONN_STRING)

stored_query = f"SELECT * FROM {OBT_TABLE_NAME}"

stored_df = pd.read_sql(
    stored_query,
    con=engine
)

print("Stored OBT data loaded from MariaDB.")
print("Table:", OBT_TABLE_NAME)
print("Shape:", stored_df.shape)

In [ ]:
display(stored_df.head())

In [ ]:
display(stored_df.info())

## Load Initial CSV Source

In [ ]:
# ============================================================
# Load Initial CSV Source
# ============================================================

raw_df = pd.read_csv(DATA_PATH)

print("Initial CSV source loaded.")
print("Raw CSV shape:", raw_df.shape)
print("Raw CSV columns:")
print(list(raw_df.columns))

In [ ]:
display(raw_df.head())

In [ ]:
source_df = raw_df.rename(columns=RENAME_MAP).copy()

print("Standardised source columns:")
print(list(source_df.columns))

In [ ]:
display(source_df.head())

## Dataset Overview

In [ ]:
# ============================================================
# Dataset Overview
# ============================================================

overview_summary = pd.DataFrame({
    "Dataset": [
        "Initial CSV source",
        "Stored MariaDB OBT table",
    ],
    "Rows": [
        raw_df.shape[0],
        stored_df.shape[0],
    ],
    "Columns": [
        raw_df.shape[1],
        stored_df.shape[1],
    ],
})

display(overview_summary)

In [ ]:
print("Stored OBT columns:")
print(list(stored_df.columns))

## Missing Value Analysis

In [ ]:
# ============================================================
# Missing Value Analysis
# ============================================================

missing_summary = pd.DataFrame({
    "Column": stored_df.columns,
    "Missing Count": stored_df.isnull().sum().values,
    "Missing Percentage": (stored_df.isnull().mean().values * 100).round(2),
})

display(missing_summary)

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(missing_summary["Column"], missing_summary["Missing Count"])
plt.xticks(rotation=90)
plt.title("Missing Values by Column in Stored OBT Data")
plt.xlabel("Column")
plt.ylabel("Missing Count")
plt.tight_layout()
plt.savefig(FIGURE_DIR + "/missing_values_by_column.png")
plt.show()

## Valid Range Check

In [ ]:
# ============================================================
# Valid Range Check
# ============================================================

range_check_results = []

for column, (minimum, maximum) in VALID_RANGES.items():
    below_min = int((stored_df[column] < minimum).sum())
    above_max = int((stored_df[column] > maximum).sum())

    range_check_results.append({
        "Column": column,
        "Expected Minimum": minimum,
        "Expected Maximum": maximum,
        "Actual Minimum": stored_df[column].min(),
        "Actual Maximum": stored_df[column].max(),
        "Values Below Minimum": below_min,
        "Values Above Maximum": above_max,
        "Passed Range Check": below_min == 0 and above_max == 0,
    })

range_check_df = pd.DataFrame(range_check_results)

display(range_check_df)

In [ ]:
blood_pressure_check = pd.DataFrame({
    "Check": ["systolic_greater_than_diastolic"],
    "Invalid Count": [int((stored_df["systolic"] <= stored_df["diastolic"]).sum())],
    "Passed": [bool((stored_df["systolic"] > stored_df["diastolic"]).all())],
})

display(blood_pressure_check)

## Descriptive Statistics

In [ ]:
# ============================================================
# Descriptive Statistics
# ============================================================

numeric_summary = stored_df[NUMERIC_FEATURE_COLUMNS].describe().T

numeric_summary["range"] = numeric_summary["max"] - numeric_summary["min"]
numeric_summary["missing_count"] = stored_df[NUMERIC_FEATURE_COLUMNS].isnull().sum()
numeric_summary["skewness"] = stored_df[NUMERIC_FEATURE_COLUMNS].skew()
numeric_summary["kurtosis"] = stored_df[NUMERIC_FEATURE_COLUMNS].kurtosis()

display(numeric_summary)

numeric_summary.to_csv(
    REPORT_DIR + "/numeric_descriptive_statistics.csv"
)

## Distribution Plots for Numerical Variables

In [ ]:
# ============================================================
# Distribution Plots for Numerical Variables
# ============================================================

for column in NUMERIC_FEATURE_COLUMNS:
    plt.figure(figsize=(7, 4))
    plt.hist(stored_df[column].dropna(), bins=30)
    plt.title(f"Distribution of {column}")
    plt.xlabel(column)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()
    plt.savefig(FIGURE_DIR + f"/distribution_{column}.png")

## Boxplots For Numerical Variables

In [ ]:
# ============================================================
# Boxplots for Outlier Visualisation
# ============================================================

for column in NUMERIC_FEATURE_COLUMNS:
    plt.figure(figsize=(7, 4))
    plt.boxplot(stored_df[column].dropna(), vert=False)
    plt.title(f"Boxplot of {column}")
    plt.xlabel(column)
    plt.tight_layout()
    plt.show()
    plt.savefig(FIGURE_DIR + f"/boxplot_{column}.png")

## Target and Gender Distribution

In [ ]:
# ============================================================
# Target and Gender Distribution
# ============================================================

target_distribution = stored_df[TARGET_COLUMN].value_counts().reindex(TARGET_ORDER)
target_percentage = stored_df[TARGET_COLUMN].value_counts(normalize=True).reindex(TARGET_ORDER) * 100

target_summary = pd.DataFrame({
    "Fitness Class": target_distribution.index,
    "Count": target_distribution.values,
    "Percentage": target_percentage.round(2).values,
})

display(target_summary)


plt.figure(figsize=(6, 4))
plt.bar(target_summary["Fitness Class"], target_summary["Count"])
plt.title("Fitness Class Distribution")
plt.xlabel("Fitness Class")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(FIGURE_DIR + "/fitness_class_distribution.png")
plt.show()

In [ ]:
gender_distribution = stored_df["gender"].value_counts()
gender_percentage = stored_df["gender"].value_counts(normalize=True) * 100

gender_summary = pd.DataFrame({
    "Gender": gender_distribution.index,
    "Count": gender_distribution.values,
    "Percentage": gender_percentage.round(2).values,
})

display(gender_summary)


plt.figure(figsize=(6, 4))
plt.bar(gender_summary["Gender"], gender_summary["Count"])
plt.title("Gender Distribution")
plt.xlabel("Gender")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(FIGURE_DIR + "/gender_distribution.png")
plt.show()

## Feature vs Feature Relationship Analysis

In [ ]:
# ============================================================
# Feature-to-Feature Relationship Analysis
# ============================================================

correlation_matrix = stored_df[NUMERIC_FEATURE_COLUMNS].corr()

display(correlation_matrix)


plt.figure(figsize=(10, 8))
plt.imshow(correlation_matrix, aspect="auto")
plt.colorbar()
plt.xticks(
    range(len(NUMERIC_FEATURE_COLUMNS)),
    NUMERIC_FEATURE_COLUMNS,
    rotation=90
)
plt.yticks(
    range(len(NUMERIC_FEATURE_COLUMNS)),
    NUMERIC_FEATURE_COLUMNS
)

In [ ]:
plt.title("Correlation Matrix of Numerical Features")
plt.tight_layout()
plt.savefig(FIGURE_DIR + "/correlation_matrix.png")
plt.show()

In [ ]:
corr_pairs = correlation_matrix.abs().where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)

top_correlations = (
    corr_pairs.stack()
    .sort_values(ascending=False)
    .reset_index()
)

top_correlations.columns = [
    "Feature 1",
    "Feature 2",
    "Absolute Correlation"
]

display(top_correlations.head(15))

## Feature vs Target Relationship

In [ ]:
# ============================================================
# Feature-to-Target Relationship using Group Means
# ============================================================

target_group_means = stored_df.groupby(TARGET_COLUMN)[NUMERIC_FEATURE_COLUMNS].mean()
target_group_means = target_group_means.reindex(TARGET_ORDER)

display(target_group_means)

In [ ]:
for column in NUMERIC_FEATURE_COLUMNS:
    plt.figure(figsize=(7, 4))

    values_by_class = [
        stored_df.loc[stored_df[TARGET_COLUMN] == class_label, column].dropna()
        for class_label in TARGET_ORDER
    ]

    plt.boxplot(values_by_class, labels=TARGET_ORDER)
    plt.title(f"{column} by Fitness Class")
    plt.xlabel("Fitness Class")
    plt.ylabel(column)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR + f"/{column}_by_fitness_class.png")
    plt.show()

## Gender vs Target Relationship

In [ ]:
# ============================================================
# Gender and Fitness Class Relationship
# ============================================================

gender_target_counts = pd.crosstab(
    stored_df["gender"],
    stored_df[TARGET_COLUMN],
    margins=True
)

gender_target_percentage = pd.crosstab(
    stored_df["gender"],
    stored_df[TARGET_COLUMN],
    normalize="index"
) * 100

display(gender_target_counts)
display(gender_target_percentage.round(2))

In [ ]:
gender_target_percentage.plot(kind="bar", figsize=(8, 5))
plt.title("Fitness Class Distribution by Gender")
plt.xlabel("Gender")
plt.ylabel("Percentage")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURE_DIR + "/fitness_class_by_gender.png")
plt.show()

## Outlier Analysis using IQR

In [ ]:
# ============================================================
# Outlier Analysis using IQR
# ============================================================

def calculate_iqr_outliers(df, columns):
    """
    Calculates outliers using the IQR rule:
    lower bound = Q1 - 1.5 * IQR
    upper bound = Q3 + 1.5 * IQR
    """

    results = []

    for column in columns:
        q1 = df[column].quantile(0.25)
        q3 = df[column].quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        outlier_mask = (
            (df[column] < lower_bound) |
            (df[column] > upper_bound)
        )

        outlier_count = int(outlier_mask.sum())
        outlier_percentage = round((outlier_count / len(df)) * 100, 2)

        results.append({
            "Column": column,
            "Q1": q1,
            "Q3": q3,
            "IQR": iqr,
            "Lower Bound": lower_bound,
            "Upper Bound": upper_bound,
            "Minimum": df[column].min(),
            "Maximum": df[column].max(),
            "Outlier Count": outlier_count,
            "Outlier Percentage": outlier_percentage
        })

    return pd.DataFrame(results)


outlier_summary = calculate_iqr_outliers(
    stored_df,
    NUMERIC_FEATURE_COLUMNS
)

display(outlier_summary)